# BirdCLEF 2026 — data preparation

Prepare manifests, fixed file-grouped folds and waveform datasets for a first baseline. This notebook uses the project package; the EDA notebook remains independent. No training, external downloads, automatic audio removal or unannotated negatives.

Select the repository `.venv` kernel after `uv sync --extra notebooks --extra deep-learning --extra audio`, as in the local starter. Run from the project root. Data and output paths come from the project config (`.env`): by default, `data/competition` and `outputs/data_prep`. Source audio is read in place.

In [1]:
from hashlib import sha256
import json

import numpy as np
import pandas as pd
import soundfile as sf
import torch
from torch.utils.data import ConcatDataset, DataLoader
from tqdm.auto import tqdm

from birdclef.config import get_config
from birdclef.runtime.environment import apply_environment_defaults
from birdclef.competition import class_columns
from birdclef.data.audio import FocalAudioDataset, SoundscapeDataset, read_audio_window
from birdclef.data.manifests import focal_manifest, soundscape_manifest
from birdclef.data.split import group_folds
from birdclef.processing.audio import log_mel

apply_environment_defaults()
config = get_config()

ROOT = config.data.competition_dir
OUTPUT = config.data.output_dir / 'data_prep'
N_FOLDS = 5
FOLD = 0
SEED = config.train.seed
RUN_HEADER_AUDIT = True

torch.manual_seed(SEED)

Detected environment: local
Loading default Config...


                                   Config                                    
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Setting                       ┃ Value                                     ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ app_name                      │ birdclef                                  │
│ logging.directory             │ C:\_GitHub\birdclef-2026\logs             │
│ logging.level                 │ INFO                                      │
│ logging.formatter             │ ColoredFormatter                          │
│ logging.date_format           │ %d-%m-%Y %H:%M:%S                         │
│ logging.full_color            │ True                                      │
│ logging.include_function      │ True                                      │
│ logging.success_level         │ 69                                        │
├───────────────────────────────┼───────────────────────────────────────────┤
│ data.competition_dir          │ C:\_GitHub\birdclef-2026\data\competition │
│ data.output_dir               │ C:\_GitHub\birdclef-2026\outputs          │
├───────────────────────────────┼───────────────────────────────────────────┤
│ split.validation_fraction     │ 0.2                                       │
│ split.seed                    │ 42                                        │
│ split.group_column            │ None                                      │
│ split.stratify                │ False                                     │
├───────────────────────────────┼───────────────────────────────────────────┤
│ tracking.project              │ birdclef-2026                             │
│ tracking.entity               │ None                                      │
│ tracking.offline              │ True                                      │
│ tracking.api_key              │ None                                      │
├───────────────────────────────┼───────────────────────────────────────────┤
│ loader.batch_size             │ 128                                       │
│ loader.val_batch_size         │ 32                                        │
│ loader.predict_batch_size     │ 32                                        │
│ loader.num_workers            │ 0                                         │
│ loader.pin_memory             │ False                                     │
│ loader.seed                   │ 0                                         │
├───────────────────────────────┼───────────────────────────────────────────┤
│ lr_scheduler.name             │ none                                      │
│ lr_scheduler.t_max            │ 10                                        │
│ lr_scheduler.min_lr           │ 0.0                                       │
│ lr_scheduler.factor           │ 0.1                                       │
│ lr_scheduler.patience         │ 2                                         │
│ lr_scheduler.monitor          │ val/loss                                  │
│ lr_scheduler.mode             │ min                                       │
├───────────────────────────────┼───────────────────────────────────────────┤
│ train.max_epochs              │ 10                                        │
│ train.seed                    │ 42                                        │
│ train.checkpoint_metric       │ val/loss                                  │
│ train.checkpoint_mode         │ min                                       │
│ train.checkpoint_top_k        │ 1                                         │
│ train.early_stopping_patience │ 3                                         │
│ train.accumulate_grad_batches │ 1                                         │
│ train.gradient_clip_val       │ 1.0                                       │
│ train.log_every_n_steps       │ 10                                        │
├───────────────────────────────┼───────────────────────────────────────────┤
│ runtime.accelerator 

## Labeled units and class order

Class order comes only from sample submission columns. Focal targets include primary and secondary labels and remain weak clip labels after cropping. Soundscape targets refer only to explicitly annotated five-second windows. Missing annotation rows never become negative examples. Identical annotation sets collapse; conflicting sets fail.

In [2]:
classes = class_columns(pd.read_csv(ROOT / 'sample_submission.csv'))
focal, focal_targets = focal_manifest(
    pd.read_csv(ROOT / 'train.csv', dtype={'primary_label': str}),
    ROOT / 'train_audio', classes,
)
windows, window_targets = soundscape_manifest(
    pd.read_csv(ROOT / 'train_soundscapes_labels.csv', dtype={'primary_label': str}),
    ROOT / 'train_soundscapes', classes,
)
windows['fold'] = group_folds(windows, n_splits=N_FOLDS)
display(pd.Series({
    'classes': len(classes), 'focal_clips': len(focal),
    'annotated_windows': len(windows), 'annotated_files': windows.filename.nunique(),
}))
display(windows.head())

classes                234
focal_clips          35549
annotated_windows      739
annotated_files         66
dtype: int64

,filename,start,end,path,row_id,fold
0,BC2026_Train_0001_S08_20250606_030007.ogg,0.0,5.0,C:\_GitHub\birdclef-2026\data\competition\trai...,BC2026_Train_0001_S08_20250606_030007_5,3
1,BC2026_Train_0001_S08_20250606_030007.ogg,5.0,10.0,C:\_GitHub\birdclef-2026\data\competition\trai...,BC2026_Train_0001_S08_20250606_030007_10,3
2,BC2026_Train_0001_S08_20250606_030007.ogg,10.0,15.0,C:\_GitHub\birdclef-2026\data\competition\trai...,BC2026_Train_0001_S08_20250606_030007_15,3
3,BC2026_Train_0001_S08_20250606_030007.ogg,15.0,20.0,C:\_GitHub\birdclef-2026\data\competition\trai...,BC2026_Train_0001_S08_20250606_030007_20,3
4,BC2026_Train_0001_S08_20250606_030007.ogg,20.0,25.0,C:\_GitHub\birdclef-2026\data\competition\trai...,BC2026_Train_0001_S08_20250606_030007_25,3


## Freeze folds and inspect AUC eligibility

Zero-based GroupKFold assignments keep all windows of a recording together. This is unstratified file CV, not a site independence guarantee. All focal recordings are available for training; no acoustic duplicate or geographic leakage audit is implied.

Save assignments before filtering or sampling. Do not average fold macro AUC scores without reporting their different evaluated class sets. Pooled OOF also compares scores from different models. Classes without negatives still block the competition metric; this table diagnoses them and does not silently change the metric.

In [3]:
support_rows = []
for fold in range(N_FOLDS):
    valid = windows.fold.eq(fold)
    assert not set(windows.loc[valid, 'filename']) & set(windows.loc[~valid, 'filename'])
    positive = window_targets.loc[valid].sum()
    negative = int(valid.sum()) - positive
    training_positive = window_targets.loc[~valid].sum()
    for label in classes:
        support_rows.append({
            'fold': fold, 'species': label,
            'train_positive_windows': int(training_positive[label]),
            'valid_positive_windows': int(positive[label]),
            'valid_negative_windows': int(negative[label]),
            'auc_defined': bool(positive[label] > 0 and negative[label] > 0),
        })
support = pd.DataFrame(support_rows)
display(support.groupby('fold').agg(
    auc_defined_classes=('auc_defined', 'sum'),
    classes_without_positives=('valid_positive_windows', lambda x: x.eq(0).sum()),
    classes_without_negatives=('valid_negative_windows', lambda x: x.eq(0).sum()),
))

,auc_defined_classes,classes_without_positives,classes_without_negatives
fold,,,
0,28,206,0
1,34,200,0
2,36,198,0
3,46,188,0
4,39,195,0


## Header audit and short-recording review

Read headers, not entire recordings. Keep every source row, including very short clips. A failed or empty audio file blocks use until repaired; a duration below one second only produces a review flag. This does not test acoustic duplicates or prove complete decodability. Set `RUN_HEADER_AUDIT=False` for a quick smoke run.

In [4]:
header_rows = []
if RUN_HEADER_AUDIT:
    sources = pd.concat([
        focal[['filename', 'path']].assign(source='focal'),
        windows[['filename', 'path']].drop_duplicates().assign(source='soundscape'),
    ], ignore_index=True)
    for row in tqdm(sources.itertuples(index=False), total=len(sources)):
        result = {'source': row.source, 'filename': row.filename}
        try:
            info = sf.info(row.path)
            result.update(duration=info.duration, sample_rate=info.samplerate,
                          channels=info.channels, frames=info.frames, error=None)
        except (OSError, RuntimeError) as error:
            result['error'] = f'{type(error).__name__}: {error}'
        header_rows.append(result)
headers = pd.DataFrame(header_rows, columns=[
    'source', 'filename', 'duration', 'sample_rate', 'channels', 'frames', 'error',
])
short_recordings = headers.loc[headers.source.eq('focal') & headers.duration.lt(1)].copy()
display(short_recordings)
display(headers.loc[headers.error.notna() | headers.frames.eq(0)])

  0%|          | 0/35615 [00:00<?, ?it/s]

,source,filename,duration,sample_rate,channels,frames,error
8,focal,1161364/iNat1264238.ogg,0.288000,32000,1,9216,None
29,focal,209233/iNat1545859.ogg,0.144000,32000,1,4608,None
39,focal,22956/iNat1693371.ogg,0.576000,32000,1,18432,None
127,focal,22973/iNat781822.ogg,0.048000,32000,1,1536,None
128,focal,22973/iNat1353836.ogg,0.096000,32000,1,3072,None
...,...,...,...,...,...,...,...
34794,focal,yecpar/iNat1360878.ogg,0.208969,32000,1,6687,None
34868,focal,yecpar/iNat1347445.ogg,0.156750,32000,1,5016,None
35006,focal,yehcar1/iNat1268927.ogg,0.017406,32000,1,557,None
35026,focal,yehcar1/iNat585409.ogg,0.072000,32000,1,2304,None


,source,filename,duration,sample_rate,channels,frames,error


## Save portable manifests and provenance

Saved CSVs use filenames rather than machine-specific absolute paths. Rebuild paths against `ROOT` after loading. Target columns follow `classes.json` exactly. SHA-256 hashes bind metadata, targets and fold assignments; the original CSV hashes record provenance. Audio content is not hashed. No source recordings are copied.

In [5]:
OUTPUT.mkdir(parents=True, exist_ok=True)
focal.drop(columns='path').to_csv(OUTPUT / 'focal.csv', index=False)
focal_targets.to_csv(OUTPUT / 'focal_targets.csv', index=False)
windows.drop(columns='path').to_csv(OUTPUT / 'soundscapes.csv', index=False)
window_targets.to_csv(OUTPUT / 'soundscape_targets.csv', index=False)
support.to_csv(OUTPUT / 'fold_class_support.csv', index=False)
headers.to_csv(OUTPUT / 'audio_headers.csv', index=False)
short_recordings.to_csv(OUTPUT / 'short_recordings_review.csv', index=False)
(OUTPUT / 'classes.json').write_text(json.dumps(classes, indent=2), encoding='utf-8')
artifact_names = [
    'focal.csv', 'focal_targets.csv', 'soundscapes.csv', 'soundscape_targets.csv',
    'fold_class_support.csv', 'classes.json', 'audio_headers.csv', 'short_recordings_review.csv',
]
provenance = {
    'version': 1, 'protocol': 'GroupKFold', 'group_column': 'filename',
    'n_folds': N_FOLDS, 'fold_numbering': 'zero-based', 'include_secondary': True,
    'header_audit_performed': RUN_HEADER_AUDIT,
    'artifacts_sha256': {name: sha256((OUTPUT / name).read_bytes()).hexdigest() for name in artifact_names},
    'source_csv_sha256': {name: sha256((ROOT / name).read_bytes()).hexdigest() for name in [
        'train.csv', 'train_soundscapes_labels.csv', 'sample_submission.csv',
    ]},
}
(OUTPUT / 'manifest.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')
if RUN_HEADER_AUDIT and (headers.error.notna().any() or headers.frames.eq(0).any()):
    raise ValueError('Audio header failures: inspect audio_headers.csv before building datasets.')
print(f'Prepared artifacts: {OUTPUT}')

Prepared artifacts: C:\_GitHub\birdclef-2026\outputs\data_prep


## Dataset and inference smoke checks

The dataset owns decoding, resampling, crops and targets. `log_mel` is an optional CPU frontend: mono 32 kHz, FFT 1024, hop 320, 128 Mel bands, 50–14,000 Hz, no centering, power dB re 1 with 80 dB range. Five seconds produce `(1, 128, 497)`. No peak normalization or training is performed.

Focal training crops are random; evaluation crops are centered; short clips are right-padded. Soundscape windows are exact and truncated windows fail. Keep loader RNG seeds fixed for reproducibility. The current numeric MLP is not an audio model and should not receive spectrograms directly.

In [6]:
train_mask = windows.fold.ne(FOLD)
valid_mask = ~train_mask
focal_dataset = FocalAudioDataset(focal, focal_targets, training=True, transform=log_mel)
sc_train = SoundscapeDataset(windows.loc[train_mask], window_targets.loc[train_mask], transform=log_mel)
sc_valid = SoundscapeDataset(windows.loc[valid_mask], window_targets.loc[valid_mask], transform=log_mel)
training = ConcatDataset([focal_dataset, sc_train])
loader = DataLoader(training, batch_size=2, shuffle=True, num_workers=0,
                    generator=torch.Generator().manual_seed(SEED))
inputs, targets = next(iter(loader))
assert inputs.shape == (2, 1, 128, 497)
assert targets.shape == (2, len(classes))
assert torch.isfinite(inputs).all()
valid_input, valid_target = sc_valid[0]
assert valid_input.shape == (1, 128, 497)
assert valid_target.shape == (len(classes),)

path = windows.path.iloc[0]
features = torch.stack([log_mel(read_audio_window(path, start=start)) for start in range(0, 60, 5)])
assert features.shape == (12, 1, 128, 497)
print({'train_examples': len(training), 'valid_windows': len(sc_valid),
       'batch_shape': tuple(inputs.shape), 'one_soundscape_shape': tuple(features.shape)})

{'train_examples': 36139, 'valid_windows': 149, 'batch_shape': (2, 1, 128, 497), 'one_soundscape_shape': (12, 1, 128, 497)}
